In [1]:
#%%


%load_ext autoreload
%autoreload 2

import json
import sys
import load_ds
from model_wrappers.cell_annotation import CellAnnotationModelWrapper
import random
import torch
import numpy as np
import os
import argparse
from pathlib import Path

sys.path.insert(0, "../")
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor

#TODO: make a model attributes file for the cell annotation model
INPUT_LAYER = "X_binned"
VOCAB_PATH = "../pretrained_models/best_model/vocab.json"
SEED = 42


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multiple GPUs
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def preprocess_data(adata, preprocessor, vocab):
    """
    Filter genes in the AnnData object based on their presence in the vocabulary.
    
    Args:
        adata (AnnData): The AnnData object containing gene expression data.
    
    Returns:
        adata (AnnData): The filtered AnnData object.
    """
    preprocessor(adata, batch_key=None)
    adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata.var.index
    ]
    return adata[:, adata.var["id_in_vocab"] >= 0]


/local/home/ktubis/PycharmProjects/scGPT/delta_tuning/../scgpt/model/model.py:22: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/local/home/ktubis/PycharmProjects/scGPT/delta_tuning/../scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [2]:
seed = 42
model_config_path = "model_configs/scgpt_pretrained_model.json"
test_data = "Muraro"
model = "../pretrained_models/best_model/best_model.pt"
max_seq_len = 301
model_name = "test"
train = True
epochs = 5
results_file = "results/test_results.json"
log_file = "logs/test_logs.txt"

In [3]:
!nvidia-smi

Fri Apr 11 11:54:29 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.54.03              Driver Version: 535.54.03    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Quadro P400                    On  | 00000000:01:00.0 Off |                  N/A |
| 34%   30C    P8              N/A /  N/A |     11MiB /  2048MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
set_seed(seed)

with open(model_config_path, "r") as f:
    config_dict = json.load(f)

vocab = GeneVocab.from_file(VOCAB_PATH)
special_tokens = [config_dict["pad_token"], "<cls>", "<eoc>"]
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

ds_loader = load_ds.PancreaticDataset()
adata_train, adata_test = ds_loader.get_train_test(test_data)

# set up the preprocessor, use the args to config the workflow
preprocessor = Preprocessor(
    use_key="X",
    normalize_total=0.0,
    binning=config_dict["n_input_bins"],
    result_binned_key=INPUT_LAYER,
)

adata_train = preprocess_data(adata_train, preprocessor, vocab)
adata_test = preprocess_data(adata_test, preprocessor, vocab)

num_celltypes = ds_loader.get_num_celltypes()
num_batches = ds_loader.get_num_batches()

/local/home/ktubis/anaconda3/envs/katyaenv/lib/python3.9/site-packages/anndata/utils.py:292: UserWarning: X converted to numpy array with dtype float64
  warnings.warn(f"{name} converted to numpy array with dtype {arr.dtype}")
/local/home/ktubis/anaconda3/envs/katyaenv/lib/python3.9/site-packages/anndata/utils.py:292: UserWarning: X converted to numpy array with dtype float64
  warnings.warn(f"{name} converted to numpy array with dtype {arr.dtype}")


scGPT - INFO - Binning data ...
scGPT - INFO - Binning data ...


In [5]:
cam = CellAnnotationModelWrapper(
    model_path=model,
    max_seq_len=max_seq_len,
    pad_value=config_dict["pad_value"],
    vocab=vocab,
    config_dict=config_dict,
    num_batches=num_batches,
    num_celltypes=num_celltypes,
    model_name=model_name,
)

print(cam.model)

/local/home/ktubis/PycharmProjects/scGPT/delta_tuning/../scgpt/model/model.py:78: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(


Error loading model, trying to load only matching parameters
TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(60697, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.2, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=512

In [ ]:

if train:
    cam.train(epochs, adata_train)

predictions, celltypes_labels, results = cam.test(adata_test, eval_batch_size=args.test_batch_size)

results_file = Path(results_file)
if not results_file.exists():
    results_file.touch()
    print("File created successfully")        

with open(results_file, "w") as f:
    json.dump(results, f)

log_file = Path(log_file)
if not log_file.exists():
    log_file.touch()
    print("Log file created successfully")

with open(log_file, "w") as f:
    f.write(f"Predictions:\n {list(predictions)}\n")
    f.write(f"Celltype Labels:\n {list(celltypes_labels)}\n")